# الفصل 56: البنية التحتية والتوسّع

> نموذج سريع على GPU واحدة لا يعني شيئًا إذا لم يستطع النظام التوسّع لخدمة آلاف المستخدمين المتزامنين بموثوقية.

- **الجزء:** 11 — النشر والتشغيل
- **الفصل:** 56
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 55: تحسين الاستدلال المتقدم
- **الوسوم:** Docker، Kubernetes، Autoscaling، GPU

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch56-infrastructure-scaling.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> نموذج سريع على GPU واحدة لا يعني شيئًا إذا لم يستطع النظام التوسّع لخدمة آلاف المستخدمين المتزامنين بموثوقية.

> **بيانات الفصل**
>
> - **الجزء:** 11 — النشر والتشغيل
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 55: تحسين الاستدلال المتقدم


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم حاويات النشر (Containerization) — الأساس الصناعي
- فهم التوسّع الأفقي عبر Kubernetes
- فهم Docker


## 1. لماذا يهمّك هذا الموضوع؟

نموذج سريع على GPU واحدة لا يعني شيئًا إذا لم يستطع النظام التوسّع لخدمة آلاف المستخدمين المتزامنين بموثوقية. هذا الفصل يغطي الطبقة الهندسية التي تحوّل نموذجًا إلى **خدمة إنتاجية حقيقية**.


## 2. الخريطة الذهنية


```text
بنية تحتية للإنتاج
├── الحاويات
│   ├── صورة CUDA + محرك الاستدلال
│   ├── تحميل النموذج (Volume/Cache)
│   └── Health/Readiness Probes
├── التنسيق
│   ├── Kubernetes Deployment
│   ├── GPU Node Pools
│   └── Pod Disruption Budget
├── التوسّع
│   ├── HPA على طول الطابور
│   ├── Cluster Autoscaler
│   └── نسخ دافئة (Warm Replicas)
├── التوازي
│   ├── Tensor Parallelism
│   ├── Pipeline Parallelism
│   └── Replicas مستقلة
└── الموثوقية
    ├── مراقبة وتنبيه
    ├── Canary/Blue-Green
    └── نسخ احتياطي وخطة تعافي
```


## 3. الشرح المتدرج

### 🟢 أ) حاويات النشر (Containerization) — الأساس الصناعي


```dockerfile
# Dockerfile إنتاجي لخدمة استدلال LLM باستخدام vLLM
FROM nvidia/cuda:12.1.0-runtime-ubuntu22.04

RUN apt-get update && apt-get install -y python3.10 python3-pip git

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# تحميل النموذج أثناء بناء الصورة (أو عبر volume mount في وقت التشغيل)
COPY ./serve_model.py .

ENV MODEL_PATH=/models/llama-3-8b-instruct
ENV TENSOR_PARALLEL_SIZE=2
ENV MAX_MODEL_LEN=8192

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["python3", "serve_model.py"]
```


```yaml
# docker-compose.yml لتجربة محلية سريعة
version: "3.8"
services:
  llm-server:
    build: .
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 2
              capabilities: [gpu]
    ports:
      - "8000:8000"
    environment:
      - NVIDIA_VISIBLE_DEVICES=0,1
    volumes:
      - ./models:/models
```


### 🟡 ب) التوسّع الأفقي عبر Kubernetes


```yaml
# kubernetes deployment كامل لخدمة LLM مع Autoscaling
apiVersion: apps/v1
kind: Deployment
metadata:
  name: llm-inference-service
spec:
  replicas: 3
  selector:
    matchLabels:
      app: llm-inference
  template:
    metadata:
      labels:
        app: llm-inference
    spec:
      containers:
      - name: vllm-server
        image: my-registry/llm-service:v1.2
        resources:
          limits:
            nvidia.com/gpu: 1
          requests:
            memory: "32Gi"
            cpu: "8"
        ports:
        - containerPort: 8000
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 60    # وقت تحميل النموذج قد يكون طويلاً
          periodSeconds: 10
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
          periodSeconds: 30
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: llm-autoscaler
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: llm-inference-service
  minReplicas: 2
  maxReplicas: 10
  metrics:
  - type: Pods
    pods:
      metric:
        name: inference_queue_length    # مقياس مخصص وليس CPU/Memory فقط!
      target:
        type: AverageValue
        averageValue: "5"
---
apiVersion: v1
kind: Service
metadata:
  name: llm-service
spec:
  selector:
    app: llm-inference
  ports:
  - port: 80
    targetPort: 8000
  type: LoadBalancer
```


> ⚠️ **خطأ شائع جدًا في Autoscaling لـ LLM**: استخدام مقاييس CPU/Memory التقليدية كمحفّز للتوسّع (scaling trigger) كما في تطبيقات الويب العادية. بالنسبة لخدمات LLM، **طول طابور الانتظار (queue length)** أو **زمن الاستجابة (latency p99)** مؤشرات أدق بكثير لأن العبء الحقيقي يقع على GPU وليس CPU، ولأن استهلاك GPU قد يكون مرتفعًا باستمرار (مشغولة) دون أن يعني ذلك بالضرورة الحاجة لمزيد من النسخ.

### 🔴 ج) طبقة التعمّق: استراتيجيات توزيع النموذج عبر عدة GPUs/عُقَد للاستدلال


In [ ]:
"""
استراتيجيات التوازي للاستدلال (تختلف عن التدريب في الفصل 21 بتركيزها على الـ latency
بدلاً من throughput التدريب):
"""

# 1) Tensor Parallelism للاستدلال — لنماذج أكبر من ذاكرة GPU واحدة
TENSOR_PARALLEL_CONFIG = {
    "tensor_parallel_size": 4,   # تقسيم كل طبقة عبر 4 GPUs
    "note": "يقلّل latency لكل طلب (كل GPU يُنفّذ جزءًا من نفس الطبقة بالتوازي)،"
            " لكن يتطلب اتصال NVLink/InfiniBand سريع جدًا بين الـ GPUs"
}

# 2) Pipeline Parallelism للاستدلال — أقل شيوعًا للاستدلال التفاعلي
PIPELINE_PARALLEL_NOTE = """
أقل ملاءمة للاستدلال منخفض الـ latency لأنه يُدخل "فقاعات" انتظار بين مراحل الأنابيب
(كل GPU ينتظر دوره)، لكنه مفيد لـ throughput عالٍ في معالجة دفعية غير تفاعلية (offline batch inference)
"""

# 3) نموذج واحد لكل GPU (Replica-based Scaling) — الأبسط والأكثر شيوعًا للتوسّع الأفقي
REPLICA_SCALING_NOTE = """
إذا كان النموذج يتّسع في ذاكرة GPU واحدة، فإن الأسلوب الأبسط والأكثر فعالية
للتعامل مع زيادة الحمل هو ببساطة تشغيل عدة نسخ (replicas) مستقلة خلف Load Balancer،
بدلاً من أي شكل من التوازي المعقّد.
"""

def choose_parallelism_strategy(model_size_gb: float, gpu_memory_gb: float,
                                   latency_sensitive: bool) -> str:
    """دليل قرار مبسّط لاختيار استراتيجية التوزيع"""
    if model_size_gb <= gpu_memory_gb * 0.7:   # هامش أمان لـ KV cache
        return "Replica-based Scaling (عدة نسخ مستقلة خلف Load Balancer)"
    elif latency_sensitive:
        return "Tensor Parallelism (لتقليل زمن الاستجابة للطلب الواحد)"
    else:
        return "Pipeline Parallelism مقبول (إن كان throughput الأولوية وليس latency)"


In [ ]:
class LoadBalancerWithHealthAwareness:
    """
    موازن تحميل بسيط يأخذ بعين الاعتبار طول طابور الانتظار الفعلي لكل نسخة،
    وليس فقط Round-Robin الساذج
    """
    def __init__(self, backend_servers: list):
        self.servers = backend_servers   # [{"url": ..., "queue_length": 0}]

    def select_server(self) -> dict:
        """اختيار الخادم الأقل ازدحامًا (Least Connections / Least Queue)"""
        return min(self.servers, key=lambda s: s["queue_length"])

    async def route_request(self, request):
        server = self.select_server()
        server["queue_length"] += 1
        try:
            response = await self._forward_request(server["url"], request)
            return response
        finally:
            server["queue_length"] -= 1

    async def _forward_request(self, url, request): ...


## 4. مثال محلول خطوة بخطوة

**المعطيات:** خدمة تستقبل 50 طلبًا/ثانية، متوسط الخرج 300 token، ونموذج 8B على A100.

**حساب الحاجة:**

\[
\text{Tokens/s} = 50 \times 300 = 15{,}000
\]

لنفترض أن نسخة واحدة تحقق 2,000 token/s عند دفعة مناسبة (رقم توضيحي يجب قياسه فعليًا):

\[
\text{Replicas} = \frac{15{,}000}{2{,}000} = 7.5 \approx 8
\]

نضيف هامشًا 30% للذروات والتعافي:

\[
8 \times 1.3 = 10.4 \approx 11 \text{ replicas}
\]

**التكلفة الشهرية التقريبية:** إذا كانت A100 تكلف 2 دولار/ساعة:

\[
11 \times 2 \times 24 \times 30 = 15{,}840 \text{ دولارًا/شهر}
\]

**بدائل لتقليل التكلفة:**
- تكميم FP8/INT8 قد يرفع throughput لكل نسخة.
- تقليل `max_tokens` إلى 200 يقلّل الحاجة إلى 8 نسخ.
- استخدام Spot Instances مع تحمل الأعطال يقلّل السعر 50-70%.
- توجيه الطلبات البسيطة إلى نموذج أصغر يقلّل الحمل على النموذج الكبير.

**الخلاصة:** التخطيط يبدأ من الحمل والـ throughput المقاس، لا من عدد المستخدمين. كل رقم يجب التحقق منه على العتاد الفعلي.


## 5. نصائح وحيل

> 💡 **اضبط Readiness Probe بعد تحميل النموذج**: إذا بدأت الحركة قبل اكتمال التحميل، ستفشل الطلبات الأولى. استخدم مسار `/health` يتحقق من جاهزية المحرك.

> 💡 **استخدم HPA على مقياس مخصص**: طول طابور الانتظار أو p95 latency أدق من CPU/GPU utilization. GPU قد تكون مشغولة بكفاءة دون حاجة لنسخة إضافية.

> 💡 **احتفظ بنسخ دافئة**: Cold Start لنموذج 8B قد يستغرق دقائق. `minReplicas` أكبر من صفر يقلّل زمن الاستجابة الأول.

> 💡 **اعزل GPU Node Pools**: لا تخلط خدمات GPU مع خدمات CPU على نفس العقد. استخدم taints/tolerations وحدود موارد واضحة.

> 💡 **اختبر التعافي**: اقتل Pod أثناء الحمل، وافصل عقدة، وتأكد أن الخدمة تستمر. الكوارث تُكتشف في الاختبار لا في الإنتاج.


## 6. أخطاء شائعة ومصائد

> ⚠️ **Autoscaling على CPU فقط**: خدمة LLM تستهلك GPU؛ CPU قد يكون منخفضًا بينما الطابور ممتلئ. استخدم queue length/latency.

> ⚠️ **نسخ كثيرة بلا حد للذاكرة**: كل نسخة تحجز ذاكرة GPU كاملة. قد تمنع نسخة جديدة من الجدولة بسبب عدم كفاية الذاكرة على العقدة.

> ⚠️ **إغفال `shm-size`**: بعض محركات الاستدلال تحتاج ذاكرة مشتركة كبيرة. الإعداد الافتراضي الصغير يسبب أعطالًا غامضة.

> ⚠️ **عدم تثبيت إصدار الصورة**: `latest` قد يجلب تغييرات غير متوافقة. استخدم tag محددًا واختبر الترقية.

> ⚠️ **نسيان تكلفة الشبكة والتخزين**: تحميل النموذج من الشبكة عند كل بدء مكلف. استخدم cache محليًا أو طبقة تخزين سريعة.


## 7. جدول مقارنة

| حجم الحمل | البنية المناسبة |
|---|---|
| تطبيق تجريبي/داخلي صغير | خادم واحد + vLLM، بدون Kubernetes |
| تطبيق إنتاجي متوسط | Kubernetes + Replica scaling + Load Balancer بسيط |
| نموذج ضخم (>GPU واحدة) | Tensor Parallelism + Replica scaling فوقه |
| حمل متقلّب بشدة (spiky traffic) | Autoscaling قائم على queue length + طبقة تخزين مؤقت (caching) |
| نشر عالمي متعدد المناطق | Multi-region deployment + Edge caching + Model routing ذكي |


## 8. ملخص أفضل الممارسات

- [ ] Health checks وReadiness probes تأخذ بعين الاعتبار وقت تحميل النموذج الطويل
- [ ] Autoscaling مبني على مقاييس خاصة بـ LLM (queue length، latency) وليس CPU فقط
- [ ] استراتيجية توزيع واضحة (replica/tensor parallel) مبنية على حجم النموذج الفعلي
- [ ] خطة Disaster Recovery وNode failure handling (خاصة لـ GPUs المكلفة والنادرة)
- [ ] فصل واضح بين بيئة staging وproduction مع نفس إعدادات GPU للاختبار الواقعي

---


## 9. أسئلة مقابلات

**سؤال 1: كيف تنشر LLM على Kubernetes بشكل صحيح؟**

> **إجابة نموذجية:** أبدأ بصورة محددة تحتوي CUDA ومحرك الاستدلال، وأحمّل النموذج من volume/cache لا من الشبكة عند كل بدء. أضبط requests/limits للـ GPU، وreadiness probe تتحقق من جاهزية المحرك، وliveness probe من استجابته. أستخدم HPA على طول الطابور أو latency، وPDB لضمان توفر النسخ أثناء الترقيات. أضيف canary deployment ومراقبة.

**سؤال 2: متى تستخدم Tensor Parallelism بدل Replicas؟**

> **إجابة نموذجية:** عندما لا يتسع النموذج في ذاكرة GPU واحدة، أو عندما يكون تقليل زمن الطلب الواحد أولوية. Tensor Parallelism يقسّم الطبقات عبر عدة GPUs ويتطلب اتصالًا سريعًا (NVLink). إذا كان النموذج يتسع في GPU واحدة، فالتوسّع بنسخ مستقلة أبسط وأرخص وأقل عرضة للاختناقات.

**سؤال 3: ما أفضل مقياس لتحفيز التوسّع؟**

> **إجابة نموذجية:** لا يوجد مقياس واحد؛ الأفضل مزيج: طول طابور الانتظار، p95 latency، وعدد الطلبات المرفوضة. CPU/GPU utilization وحدها مضللة. أستخدم مقياسًا مركبًا مع حدود واضحة، وأختبر أن التوسّع يتحقق قبل أن يتدهور p99.


## 10. تمرين عملي

**الهدف:** كتابة خطة نشر لخدمة 20 طلبًا/ثانية بزمن p95 أقل من 2 ثانية.

**الخطوات:**
1. قِس throughput وزمن الاستجابة لنسخة واحدة على العتاد المستهدف.
2. احسب عدد النسخ المطلوبة مع هامش 30%.
3. حدّد requests/limits وreadiness probe.
4. اكتب HPA بمقياس queue length، وحدّد min/max replicas.
5. احسب التكلفة الشهرية بافتراض سعر GPU/ساعة.
6. صمّم خطة canary: 10% ثم 50% ثم 100%.
7. اكتب 3 سيناريوهات فشل وخطة التعامل معها.

**معيار النجاح:** خطة قابلة للتنفيذ بأرقام واضحة، وليس قائمة مصطلحات.


## 11. مراجع للتعمّق

- Kubernetes Documentation. *Horizontal Pod Autoscaling* and *GPU Scheduling*. kubernetes.io/docs.
- vLLM. *Production Deployment Guide*. docs.vllm.ai.
- NVIDIA. *Triton Inference Server Documentation*. docs.nvidia.com.
- Ray. *Ray Serve Documentation*. docs.ray.io.
- Dean, J., & Barroso, L. A. (2013). *The Tail at Scale*. Communications of the ACM.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النشر والتشغيل / الفصل 56: البنية التحتية والتوسّع.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
